<a href="https://colab.research.google.com/github/jagritbhatia/sentiment-analysis-model/blob/main/sentiment_analysis_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets evaluate -q

import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate


csv_path = "/content/drive/MyDrive/ME/IMDB Dataset.csv"


df = pd.read_csv(csv_path)


dataset = Dataset.from_pandas(df)


def label_to_int(example):
    example['label'] = 1 if example['sentiment'] == 'positive' else 0
    return example

dataset = dataset.map(label_to_int)


dataset = dataset.train_test_split(test_size=0.2)


model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization with labels
def preprocess_function(examples):
    tokens = tokenizer(examples['review'], padding='max_length', truncation=True)
    tokens['labels'] = examples['label']
    return tokens

tokenized_datasets = dataset.map(preprocess_function, batched=True)


model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
)


metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return metric.compute(predictions=predictions, references=labels)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics
)


trainer.train()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.208700,0.183725,0.934400
2,0.133000,0.234668,0.933900
3,0.072300,0.280502,0.936900


TrainOutput(global_step=7500, training_loss=0.15194558970133465, metrics={'train_runtime': 5985.3954, 'train_samples_per_second': 20.049, 'train_steps_per_second': 1.253, 'total_flos': 1.589608783872e+16, 'train_loss': 0.15194558970133465, 'epoch': 3.0})

In [2]:
model.save_pretrained("/content/drive/MyDrive/my_models/distilbert_finetuned")
tokenizer.save_pretrained("/content/drive/MyDrive/my_models/distilbert_finetuned")


('/content/drive/MyDrive/my_models/distilbert_finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/my_models/distilbert_finetuned/special_tokens_map.json',
 '/content/drive/MyDrive/my_models/distilbert_finetuned/vocab.txt',
 '/content/drive/MyDrive/my_models/distilbert_finetuned/added_tokens.json',
 '/content/drive/MyDrive/my_models/distilbert_finetuned/tokenizer.json')

In [5]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import evaluate

model_path = "/content/drive/MyDrive/my_models/distilbert_finetuned"

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

metric = evaluate.load("accuracy")

def evaluate_model(eval_dataset):
    model.eval()
    all_logits = []
    all_labels = []

    for sample in eval_dataset:
        inputs = tokenizer(sample["review"], padding=True, truncation=True, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        all_logits.append(logits)


        label_tensor = torch.tensor([sample["label"]])
        all_labels.append(label_tensor)

    logits = torch.cat(all_logits, dim=0)
    predictions = torch.argmax(logits, dim=1)
    labels = torch.cat(all_labels, dim=0)

    acc = metric.compute(predictions=predictions, references=labels)
    return acc

def predict_sentiment(review_text):
    inputs = tokenizer(review_text, padding=True, truncation=True, max_length=512, return_tensors="pt")
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()
    return "Positive" if predicted_class_id == 1 else "Negative"






In [6]:
accuracy = evaluate_model(tokenized_datasets['test'])
print("Test Accuracy:", accuracy)

new_review = "The movie was fantastic and I loved it!"
print(predict_sentiment(new_review))

Test Accuracy: {'accuracy': 0.9369}
Positive
